In [ ]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import wfdb
#from matplotlib.axes._axes import _log as matplotlib_axes_logger
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, BatchNormalization, ReLU, Conv1DTranspose, Concatenate
from tensorflow.keras.callbacks import ModelCheckpoint

In [ ]:
input_dir= "input"
assert os.path.isdir(input_dir), f"{input_dir} directory with the dataset is missing"
input_dataset = input_dir+"/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/records500"
ver = input_dataset.find("1.0.")
print(f"Using dataset version {input_dataset[ver:ver + 5]}")

In [ ]:
file_paths=[]
for dirpath, dirnames, filenames in os.walk(input_dataset):
    for file in filenames:
        if file.endswith(".dat"):
            file_path = os.path.join(dirpath, file)
            file_paths.append(file_path)

In [ ]:
file_paths = np.asarray(file_paths)

In [ ]:
file_paths.shape

In [ ]:
# Suppose file_paths is your list of all files
# Example: file_paths = ["/path/file1", "/path/file2", ...]

# First: train (70%) + temp (30%)
train_paths, temp_paths = train_test_split(
    file_paths, test_size=0.30, random_state=42, shuffle=True
)

# Then: split temp into validation (10%) and test (20%)
# validation is 1/3 of temp (10/30), test is 2/3 (20/30)
val_paths, test_paths = train_test_split(
    temp_paths, test_size=2/3, random_state=42, shuffle=True
)

print(f"Train: {len(train_paths)} files")
print(f"Val:   {len(val_paths)} files")
print(f"Test:  {len(test_paths)} files")

In [ ]:
def load_challenge_data(filename):
    x , _= wfdb.rdsamp(filename.strip(".dat"))
    return x

In [ ]:
load_challenge_data(train_paths[0])

In [ ]:
lead_labels = [
    'I',  'II', 'III',
    'aVR','aVL','aVF',
    'V1', 'V2', 'V3',
    'V4', 'V5', 'V6',
    'V1', 'II', 'V5'
]

In [ ]:
quartile = 1250

In [ ]:
# ---- Batch Generators ----
def batch_generator(batch_size, gen_ecg):
    """
    Yields batches of (features, labels).
    """
    batch_input = np.zeros((batch_size, 5000, 15))
    batch_output = np.zeros((batch_size, 5000, 12))

    while True:
        for i in range(batch_size):
            batch_output[i], batch_input[i] = next(gen_ecg)

        yield batch_input, batch_output

In [ ]:
def generate_io(files, quartile):
    """
    Loads and preprocesses ECG data in shuffled order.
    """
    while True:
        for i in files:
            data = load_challenge_data(i)
            X_processed = process_ecg(data, quartile)
            yield np.expand_dims(data,0), X_processed

In [ ]:
# ---- ECG Preprocessing ----
def process_ecg(X_train, quartile):
    """
    Apply channel masking and zeroing according to quartile rules.
    """
    if len(X_train.shape) == 2:
        X_train = np.expand_dims(X_train, 0)
    T = X_train.shape[1]  # time length
    X_train_input = X_train[:,: ,[0,1,2,3,4,5,6,7,8,9,10,11,6,1,10]]

    # zero out chunks by quartiles
    X_train_input[:, int(quartile):,:3] = 0

    idx1 = np.r_[0:int(quartile), int(quartile*2):T]
    X_train_input[:, idx1,3:6] = 0

    idx2 = np.r_[:int(quartile*2), int(quartile*3):T]
    X_train_input[:, idx2, 6:9] = 0

    X_train_input[:, :int(quartile*3), 9:12] = 0

    return X_train_input

In [ ]:
test_gen = generate_io(train_paths, quartile)

In [ ]:
a,b = next(test_gen)

In [ ]:
a.shape

In [ ]:
def plot_ecg_waveform(waveform_arr, length):
    '''Plot the waveforms for ed-bwh-ecg dataset.

    Parameter: np.array with shape=(15,5000)

    Each row of the waveform array contains the numeric waveform for one of the leads. The
    first 12 rows contain short duration leads. The last 3 rows contain long duration leads.
    '''

    global lead_labels

    fig, axes = plt.subplots(15, 1, sharex=True)
    fig.set_figwidth(10)
    fig.set_figheight(10)

    for ax, lead_waveform, lead_name in zip(axes, waveform_arr, lead_labels):
        ax.plot(lead_waveform, label=lead_name)
        ax.set_xlim((0,length))
        ax.legend(loc='center left')
        xaxis = ax.get_shared_x_axes()

    fig.show()

In [ ]:
#plot_ecg_waveform(n,X_train.shape[2])

In [ ]:
# Self-Attention Layer
import keras
@keras.saving.register_keras_serializable()  # This makes it loadable
class SelfAttention1D(layers.Layer):
    def __init__(self, channels, **kwargs):
        super(SelfAttention1D, self).__init__(**kwargs)  # <-- handles trainable, name, dtype
        self.channels = channels

        # Define layers here instead of in build()
        self.query_dense = layers.Dense(channels)
        self.key_dense = layers.Dense(channels)
        self.value_dense = layers.Dense(channels)
        self.softmax = layers.Softmax(axis=-1)

    def call(self, inputs):
        # QKV projections
        query = self.query_dense(inputs)
        key = self.key_dense(inputs)
        value = self.value_dense(inputs)

        # Attention weights
        score = tf.matmul(query, key, transpose_b=True)
        score = score / tf.sqrt(tf.cast(self.channels, tf.float32))
        weights = self.softmax(score)

        # Apply attention
        output = tf.matmul(weights, value)
        return output

    def get_config(self):
        config = super(SelfAttention1D, self).get_config()
        config.update({
            "channels": self.channels,
        })
        return config

# U-Net Encoder Block
def encoder_block(inputs, filters):
    conv = layers.Conv1D(filters, kernel_size=3, strides=1, padding='same', activation='relu')(inputs)
    conv = layers.Conv1D(filters, kernel_size=3, strides=1, padding='same', activation='relu')(conv)
    pool = layers.MaxPooling1D(pool_size=2)(conv)
    return conv, pool

# U-Net Decoder Block
def decoder_block(inputs, skip, filters):
    up = layers.Conv1DTranspose(filters, kernel_size=2, strides=2, padding='same')(inputs)

    if skip is not None:
        # Compute difference in length
        diff = skip.shape[1] - up.shape[1]
        if diff > 0:
            # Crop skip to match upsampled tensor
            skip = layers.Cropping1D(cropping=(0, diff))(skip)
        elif diff < 0:
            # Crop upsampled tensor if needed
            up = layers.Cropping1D(cropping=(0, -diff))(up)

        merge = layers.Concatenate()([up, skip])
    else:
        merge = up
    conv = layers.Conv1D(filters, kernel_size=3, strides=1, padding='same', activation='relu')(merge)
    conv = layers.Conv1D(filters, kernel_size=3, strides=1, padding='same', activation='relu')(conv)

    return conv

# Full U-Net Model with Self-Attention in Bottleneck
def unet_1d_with_self_attention(input_shape=(1000, 15)):
    inputs = layers.Input(input_shape)
    # Encoder
    enc1, pool1 = encoder_block(inputs, 64)
    enc2, pool2 = encoder_block(pool1, 128)
    enc3, pool3 = encoder_block(pool2, 256)
    enc4, pool4 = encoder_block(pool3, 512)
    # Bottleneck with Self-Attention
    bottleneck = layers.Conv1D(1024, kernel_size=3, padding='same', activation='relu')(pool4)
    bottleneck = layers.Conv1D(1024, kernel_size=3, padding='same', activation='relu')(bottleneck)
    bottleneck = SelfAttention1D(1024)(bottleneck)
    # Decoder
    dec4 = decoder_block(bottleneck, enc4, 512)
    dec3 = decoder_block(dec4, enc3, 256)
    dec2 = decoder_block(dec3, enc2, 128)
    dec1 = decoder_block(dec2, enc1, 64)
    # Adjust time steps to exactly 1000
    current_time_steps = dec1.shape[1]  # Should be 992
    padding_needed = 1000 - current_time_steps  # Should be 8
    if padding_needed > 0:
        dec1 = layers.ZeroPadding1D(padding=(0, padding_needed))(dec1)  # Pad to reach 1000
    elif padding_needed < 0:
        dec1 = layers.Cropping1D(cropping=(0, -padding_needed))(dec1)  # Crop if needed
    # Output
    outputs = layers.Conv1D(12, kernel_size=1, activation='linear')(dec1)
    return Model(inputs, outputs)

# Create and summarize the model
model = unet_1d_with_self_attention(input_shape=(1000, 15))
model.summary()

In [ ]:
def build_unet_ecg(input_shape=(5000, 15)):
    # Input: (samples, 1024 time steps, 3 leads [II, V1, V5])
    inputs = Input(shape=input_shape)

    # --------------------- Encoder ---------------------
    # Block 1
    e1 = Conv1D(64, 3, padding='same')(inputs)
    e1 = BatchNormalization()(e1)
    e1 = ReLU()(e1)
    p1 = MaxPooling1D(2)(e1)  # Output: 512

    # Block 2
    e2 = Conv1D(128, 3, padding='same')(p1)
    e2 = BatchNormalization()(e2)
    e2 = ReLU()(e2)
    p2 = MaxPooling1D(2)(e2)  # Output: 256

    # Block 3
    e3 = Conv1D(256, 3, padding='same')(p2)
    e3 = BatchNormalization()(e3)
    e3 = ReLU()(e3)
    p3 = MaxPooling1D(2)(e3)  # Output: 128

    # Block 4 (Bottleneck)
    e4 = Conv1D(512, 3, padding='same')(p3)
    e4 = BatchNormalization()(e4)
    e4 = ReLU()(e4)
    e4 = SelfAttention1D(512)(e4)

    # --------------------- Decoder ---------------------
    # Block 1 (Upsample to 128)
    d1 = Conv1DTranspose(256, 3, strides=2, padding='same')(e4)
    d1 = Concatenate()([d1, e3])  # Skip connection from e3
    d1 = Conv1D(256, 3, padding='same')(d1)
    d1 = BatchNormalization()(d1)
    d1 = ReLU()(d1)

    # Block 2 (Upsample to 256)
    d2 = Conv1DTranspose(128, 3, strides=2, padding='same')(d1)
    d2 = Concatenate()([d2, e2])  # Skip connection from e2
    d2 = Conv1D(128, 3, padding='same')(d2)
    d2 = BatchNormalization()(d2)
    d2 = ReLU()(d2)

    # Block 3 (Upsample to 512)
    d3 = Conv1DTranspose(64, 3, strides=2, padding='same')(d2)
    d3 = Concatenate()([d3, e1])  # Skip connection from e1
    d3 = Conv1D(64, 3, padding='same')(d3)
    d3 = BatchNormalization()(d3)
    d3 = ReLU()(d3)

    # Final Upsample to 1024 (Original length)
    #d4 = Conv1DTranspose(32, 3, strides=2, padding='same')(d3)
    #d4 = Conv1D(32, 3, padding='same')(d4)
    #d4 = BatchNormalization()(d4)
    #d4 = ReLU()(d4)

    # Output Layer: 9 leads (I, III, aVR, aVL, aVF, V2, V3, V4, V6)
    outputs = Conv1D(12, 1, activation='linear')(d3)  # 1x1 conv to match channels

    model = Model(inputs, outputs)
    return model

In [ ]:
# Initialize and compile the model
model = build_unet_ecg(b.shape[1:])
#model = unet_1d_with_self_attention()
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-4),
    loss='mse',
    metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

model.summary()  # Verify architecture shapes

In [ ]:
bg = batch_generator(32, generate_io(train_paths,quartile))

In [ ]:
next(bg)

In [ ]:
# Create output directory for trained models and other files
output_dir="output"
output_model_dir="models"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(output_dir+"/"+output_model_dir, exist_ok=True)

In [ ]:
# Define the checkpoint callback
checkpoint = ModelCheckpoint(
    output_model_dir+"/best_ecg_reconstruction_model.keras",  # path to save the best model
    monitor='val_loss',                  # metric to monitor
    verbose=1,                           # print messages
    save_best_only=True,                 # save only the best model
    mode='min'                           # 'min' because lower val_loss is better
)

# Train the model with the checkpoint
history = model.fit(
    batch_generator(32, generate_io(train_paths,quartile)),
    steps_per_epoch=int(len(train_paths)/32),
    validation_data = batch_generator(32, generate_io(val_paths,quartile)),
    validation_steps=len(val_paths) // 32,
    epochs=100,
    batch_size=32,
    callbacks=[checkpoint]               # pass the checkpoint callback here
)

from tensorflow.keras.models import load_model

best_model = load_model(
    output_model_dir+"/best_ecg_reconstruction_model.keras",
    custom_objects={"SelfAttention1D": SelfAttention1D}
)
# (Optional) Save it again with a different name if you want
#best_model.save("ecg_reconstruction_model_best.h5")

In [ ]:
# Extract loss values
train_loss = history.history['loss']
val_loss = history.history['val_loss']

# Plot
plt.figure(figsize=(8, 5))
plt.plot(train_loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
test_data_out = []
test_data_in = []
for i in test_paths:
    output_data = load_challenge_data(i)
    input_data = process_ecg(output_data, quartile)
    input_data = input_data[0]
    test_data_out.append(output_data)
    test_data_in.append(input_data)
test_data_out = np.asarray(test_data_out)
test_data_in = np.asarray(test_data_in)

In [ ]:
from scipy.stats import pearsonr

# Predict on test set
Y_pred = best_model.predict(test_data_in)

In [ ]:
np.savez(output_dir+"/data.npy", original=test_data_out, reconstructed=Y_pred)

In [ ]:
# Compute Pearson Correlation Coefficient for each lead
pcc_scores = [pearsonr(test_data_out[:, :, i].flatten(), Y_pred[:, :, i].flatten())[0] for i in range(test_data_out.shape[2])]

print(f"Mean PCC across leads: {np.mean(pcc_scores):.4f}")

print("PCC pr lead:")
for num, score in enumerate(pcc_scores):
    print(lead_labels[num] + ": ",score )

In [ ]:
# Compute RMSE for each lead
rmse_scores = [np.sqrt(mean_squared_error(test_data_out[:, :, i].flatten(),
                                          Y_pred[:, :, i].flatten()))
               for i in range(test_data_out.shape[2])]

print(f"Mean RMSE across leads: {np.mean(rmse_scores):.4f}")
print("\nRMSE per lead:")
for num, score in enumerate(rmse_scores):
    print(f"{lead_labels[num]}: {score:.4f}")

In [ ]:
# Correction based on leads we know
Y_pred[:,:,[1,6,10]] = test_data_out[:,:,[1,6,10]]

In [ ]:
# Compute Pearson Correlation Coefficient for each lead
pcc_scores = [pearsonr(test_data_out[:, :, i].flatten(), Y_pred[:, :, i].flatten())[0] for i in range(test_data_out.shape[2])]

print(f"Mean PCC across leads: {np.mean(pcc_scores):.4f}")

print("PCC pr lead:")
for num, score in enumerate(pcc_scores):
    print(lead_labels[num] + ": ",score )

In [ ]:
# Compute RMSE for each lead
rmse_scores = [np.sqrt(mean_squared_error(test_data_out[:, :, i].flatten(),
                                          Y_pred[:, :, i].flatten()))
               for i in range(test_data_out.shape[2])]

print(f"Mean RMSE across leads: {np.mean(rmse_scores):.4f}")
print("\nRMSE per lead:")
for num, score in enumerate(rmse_scores):
    print(f"{lead_labels[num]}: {score:.4f}")

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(20, 10))
fig.suptitle("4x3 Subplots with Sine Waves", fontsize=16)

# Loop through the 3x3 grid and plot sine waves with different frequencies
cnt = 0
for i in range(4):
    for j in range(3):
        ax = axes[i, j]
        ax.set_title(lead_labels[cnt])  # <-- set subplot title here
        ax.plot(Y_pred[0,:,cnt], label="Reconstructed")
        ax.plot(test_data_out[0,:,cnt], label="Original")
        ax.grid(True)
        cnt += 1

# Adjust layout to prevent overlapping
plt.tight_layout(rect=(0.0, 0.0, 1.0, 0.96))
plt.legend()
# Show plot
plt.show()